# Suite SVS — SystemVerilog simplified by IEEE 1800-2023

`SystemVerilog.Simplifiers.OfAny(term, scope)` simplifies a SystemVerilog term by IEEE 1800-2023's rules: 4-state
values through every operator, 11.6's sizing (context-determined and self-determined operands, sign extension) and
11.8's signedness. What the scope decides folds into the literal of its value, and the rest stays, the residual, each
subterm folded wherever its literal means what it did there: a self-determined subterm alone, a context-determined one
once its context's type is known, from the whole expression, the scope's values and the types it declares for the names
it does not bind. What has no rule stays too, named with its class in the evaluation inventory (`residues`): a write, a
class, a simulator's state (`d`), what waits on domain inference (`c`), what has no value of its own (`b`). Terms are
written through the dialect's writers and builders, in a transaction (`term`); `text` gives a residual's source, as the
printer writes it, and `shown` a value as its language writes it (a `Logic` as SystemVerilog writes a vector).

In [ ]:
import math

from mbse.Expressions import Framework
from mbse.Expressions.IEEE754 import Expressions as IEEE754
from mbse.Expressions.ISOIEC11404 import Expressions as ISOIEC11404
from mbse.Expressions.SystemVerilog import Assertions as SVAssertions, Constraints as SVConstraints, \
    Expressions as SystemVerilog, Patterns as SVPatterns, Preprocessor as SVPreprocessor, Types as SVTypes
from mbse.Expressions.SystemVerilog.Expressions import Domains as SVD, Formats, Simplifiers as SVV, Expressions as SV, \
    Text as SVText
from mbse.Expressions.Framework import Terms as F
from mbse.Expressions.Framework.Simplifiers import NotEvaluated
from mbse.Schemas.Framework import Builders
from support import raises

store = Builders.Store()
for install in [Framework.install, ISOIEC11404.install, IEEE754.install, SystemVerilog.install, SVTypes.install,
                SVPatterns.install, SVAssertions.install, SVConstraints.install, SVPreprocessor.install]:
    store.transaction(install)
store.transaction(SystemVerilog.define)  # SystemVerilog's types, which its rules read through a term's holder


def ports(txn):
    """A port, a reference object: a class handle the scope may bind."""
    S = txn.Schemas
    return txn.uid("svs.ports/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Port": S.OfObject})
        .spec([lambda _: _.Port.name("Port").ref(True).properties({"width": S.OfInt})])]))])


store.transaction(ports)
with store.transaction():
    port = store.Port(lambda p: p.width(8))


def term(spec):
    """The SystemVerilog term `spec` writes, a native value as its literal, the root of a holder built in a
    transaction."""
    with store.transaction():
        return F.term(store.Expressions.Framework.Expression(lambda e: e.root(SV.DIALECT.spec(spec))))


def simplified(spec, scope=None):
    """The simplification of the term `spec` writes, in `scope` (a `Scope`, or a mapping of variables)."""
    return SVV.OfAny(term(spec), scope)


def text(spec, scope=None):
    """The source of the residual of the term `spec` writes, simplified in `scope`."""
    return SVText.ToText(term(simplified(spec, scope).residual))


def shown(value):
    """A value as its language writes it: a `Logic` as SystemVerilog writes a vector (`4'b10x1`)."""
    return repr(value) if isinstance(value, SVD.Logic) else value


def value(spec, scope=None):
    """The value the term `spec` writes simplifies to, in `scope`, as `shown` writes it: its literal root's."""
    return shown(simplified(spec, scope).evaluate())


def rule(spec, scope=None):
    """The value SystemVerilog's folding rules give a term where no literal spells it (an array's): the term stays."""
    scope = scope if isinstance(scope, SVV.Scope) else SVV.Scope(scope or {})
    found = SVV.RULES.evaluate(term(spec), scope)
    return [shown(item) for item in found] if isinstance(found, list) else shown(found)


def reasons(spec, scope=None):
    """Why what stayed in the residual stayed: each residue's reason and class."""
    return [(residue.reason, residue.category) for residue in simplified(spec, scope).residues]


def integer(n, width=32, signed=True):
    """The value of `n` as a vector of `width` bits, as `shown` writes it: `integer`'s by default."""
    return f"{width}'{'s' if signed else ''}b{n % (1 << width):0{width}b}"


x, y, q, s, p = (SV.identifier(name) for name in ("x", "y", "q", "s", "p"))
b, u, v, call = SV.binary, SV.unary, SV.vector, SV.call
INTEGER, INT, REAL, SHORTREAL = (SVD.named(store, name) for name in ("integer", "int", "real", "shortreal"))
atom = lambda keyword, signing=None: lambda t: t.integeratomtype(  # noqa: E731: `int`, `int unsigned`
    lambda a: a.keyword(keyword) if signing is None else a.keyword(keyword).signing(signing))
dollar = lambda u_: u_.dollarexpression(lambda d: d)  # noqa: E731: `$`
null = lambda u_: u_.constant(lambda r: r.singleton("null"))  # noqa: E731
fill = lambda bit: lambda u_: u_.constant(lambda r: r.fill(bit))  # noqa: E731: '0, '1, 'x, 'z
this = lambda u_: u_.thisexpression(lambda t: t)  # noqa: E731
parenthesized = lambda inner: lambda u_: u_.parenthesizedexpression(lambda r: r.expression(SV.DIALECT.spec(inner)))  # noqa: E731


def time(magnitude, unit):
    """A time literal, `10ns`."""
    return lambda u_: u_.literal(lambda r: r.value(lambda m: m.as_scalar(magnitude)).unit(unit))


def scoped(package, name):
    """A package's scoped name, `package::name`; `$unit::name` where the package is None, `p::q::name` a pair."""
    def scope(i):
        if package is None:
            return i.unitname(lambda d: d)
        if isinstance(package, tuple):
            return i.scopedname(lambda c: c.scope(lambda j: j.identifier(lambda d: d.spelling(package[0]))).name(
                lambda j: j.identifier(lambda d: d.spelling(package[1]))))
        return i.identifier(lambda d: d.spelling(package))
    return lambda u_: u_.nameexpression(lambda r: r.name(lambda n: n.scopedname(lambda c: c.scope(scope).name(
        lambda i: i.identifier(lambda d: d.spelling(name))))))


def vector_type(keyword, left, right, signing=None):
    """A vector type of one packed dimension, `logic [left:right]`, signed or not."""
    def write(t):
        def record(n):
            n.keyword(keyword).dimensions([lambda d: d.rangedimension(lambda r: r.left(SV.literal(left)).right(
                SV.literal(right)))])
            return n if signing is None else n.signing(signing)
        return t.integervectortype(record)
    return write

<a id="SVS-01"></a>
## SVS-01 · Literals, names, `this`, `null` and a package's names fold into the literal of their value; a time literal by the scope's time unit

In [ ]:
# The root folds into its type's literal: an `integer`'s an unsized decimal integer, any other integral value a sized
# vector of its bits and signedness, a real a real; a literal stays as written.
assert text(b("+", 1, b("*", x, 2)), {"x": 3}) == "7" and text(b("-", 1, 8)) == "-7"
assert text(b("+", v("0011"), v("0001"))) == "4'b0100" and text(b("*", 1.5, 2)) == "3.0" and text(v("10x1")) == "4'b10x1"
assert value(x, {"x": True}) == "1'b1" and value(x, {"x": 2 ** 40}) == integer(2 ** 40, 64)  # a variable's type is its value's
# `null` is the null class handle, which compares with handles by identity; `this` is the scope's, by that name.
assert simplified(null).literal and simplified(null).evaluate() is None and text(null) == "null"
assert text(b("==", x, y), {"x": None}) == "null == y"  # a name's value, `null`, is its literal
handle = {"count": 4}
assert value(b("==", x, null), {"x": None}) == "1'b1" and value(b("!=", x, null), {"x": handle}) == "1'b1"
assert value(b("===", x, y), {"x": handle, "y": handle}) == "1'b1" and value(b("==", x, y), {"x": handle, "y": {"count": 4}}) == "1'b0"
assert value(SV.member(this, "count"), {"this": handle}) == integer(4) and text(SV.member(this, "count")) == "this.count"
assert value(b("==", x, x), {"x": port}) == "1'b1" and value(SV.member(this, "width"), {"this": port}) == integer(8)  # data
for spec in (b("==", x, 1), b("==", 1, x)):  # a handle is no number
    with raises(TypeError, match="== expects a number, got dict"):
        value(spec, {"x": handle})
# A package's names are its scope's (`packages`), `$unit::x` among them; one the scope does not provide stays, named.
package = SVV.Scope({"width": 8}, {"twice": lambda n: 2 * n.integer()})
calls = lambda u_: u_.callexpression(lambda r: r.callee(scoped("pkg", "twice")).arguments([SV.literal(21)]))  # noqa: E731
assert value(b("+", scoped("pkg", "width"), 1), SVV.Scope(packages={"pkg": package})) == integer(9)
assert value(calls, SVV.Scope(packages={"pkg": package})) == integer(42)
assert value(scoped(None, "width"), SVV.Scope(packages={"$unit": package})) == integer(8) and text(scoped(None, "width")) == "$unit::width"
assert reasons(scoped("pkg", "width")) == [("SystemVerilog scopedname is not simplified", "d")]  # a class's, or unknown
assert reasons(scoped(("p", "q"), "x"), SVV.Scope(packages={"p": package})) == [("SystemVerilog scopedname is not simplified", "d")]
assert text(b("+", y, scoped("pkg", "missing")), SVV.Scope(packages={"pkg": package})) == "y + pkg::missing"
# A time literal is a `realtime` in the scope's time unit, rounded to its precision (3.14.2): 1, 10 or 100 of a unit.
assert value(time(1.234, "ns"), SVV.Scope(timeunit="1ns", timeprecision="1ps")) == 1.234
assert value(time(1.23456, "ns"), SVV.Scope(timeunit="1ns", timeprecision="10ps")) == 1.23
assert value(time(1.5, "us"), SVV.Scope(timeunit="100ps")) == 15000.0 and value(time(3.0, "fs"), SVV.Scope(timeunit="1ns")) == 0.0
assert value(b("+", time(2.0, "ns"), 1), SVV.Scope(timeunit="1ns")) == 3.0
assert reasons(time(10.0, "ns")) == [("SystemVerilog time literal without a time unit is not simplified", "a")]
for options, message in [({"timeunit": "2ns"}, "a time unit is 1, 10 or 100 of s, ms, us, ns, ps or fs, got '2ns'"),
                         ({"timeunit": "1ns", "timeprecision": "10ns"}, "a time precision is at most its unit, got 10ns for 1ns"),
                         ({"timeprecision": "1ps"}, "a time precision needs a time unit"),
                         ({"mintypmax": "all"}, "min:typ:max's choice is min, typ or max, got 'all'")]:
    with raises(ValueError, match=message):
        SVV.Scope(**options)

<a id="SVS-02"></a>
## SVS-02 · Subterms fold where their meaning is kept: a self-determined one alone, a context-determined one once its context's type is known

In [ ]:
# A self-determined subterm folds alone: a concatenation's operand, a shift's count, an index, a condition, a cast's
# operand, a function's argument, a reduction's operand.
assert text(SV.concatenation(y, b("+", v("0001"), v("0001")))) == "{y, 4'b0010}"
assert text(b("<<", y, b("+", 1, 2))) == "y << 3" and text(SV.select(y, b("+", 1, 1))) == "y[2]"
assert text(SV.conditional(b("<", 1, 2), y, x)) == "1'b1 ? y : x" and text(call("f", b("+", 1, 2), y)) == "f(3, y)"
assert text(u("&", b("|", v("01"), v("10")))) == "1'b1" and text(b("&&", u("|", y), b("<", 1, 2))) == "|y && 1'b1"
# A context-determined one is computed in its context's type (11.6), so it folds once that type is known: here `y`'s,
# which the scope declares (`types`); unknown, `3 * 2` stays, since `y` could widen it.
assert text(b("+", y, b("*", 3, 2))) == "y + 3 * 2"
assert text(b("+", y, b("*", 3, 2)), SVV.Scope(types={"y": INTEGER})) == "y + 6"
assert text(b("+", y, b("+", 2147483647, 1)), SVV.Scope(types={"y": SVD.vector(64)})) == "y + 64'b" + "0" * 32 + "1" + "0" * 31
assert text(b("+", y, b("-", 3, 8)), SVV.Scope(types={"y": SVD.vector(64)})) == "y + 64'b" + "1" * 61 + "011"  # unsigned
assert text(b("==", y, b("+", v("1111"), v("0001"))), SVV.Scope(types={"y": SVD.vector(5)})) == "y == 5'b10000"
assert text(b("+", y, b("+", 1, 2)), SVV.Scope(types={"y": SVD.vector(8)})) == "y + 32'b" + "0" * 30 + "11"
# A type with no literal (2-state, `shortreal`) folds into a literal where its states cannot matter; where they could,
# into a cast of a literal to its type, which keeps them (SVS-15).
assert text(b("+", SV.cast(atom("int"), b("+", 1, 2)), y), SVV.Scope(types={"y": INT})) == "int'(3) + y"
assert text(b("+", SV.cast(atom("int"), b("+", 1, 2)), y), SVV.Scope(types={"y": INTEGER})) == "3 + y"
assert text(b("&&", SV.cast(atom("int"), 3), y)) == "32'sb" + "0" * 30 + "11 && y"
assert text(b("+", x, y), {"x": True}) == "bit'(1'b1) + y" and text(b("&&", x, y), {"x": True}) == "1'b1 && y"  # a `bit`
assert text(b("+", u("-", b("+", 1, 2)), y), SVV.Scope(types={"y": INTEGER})) == "-3 + y"
assert text(b("+", parenthesized(b("+", 1, 2)), y), SVV.Scope(types={"y": INTEGER})) == "3 + y"
assert text(call("f", parenthesized(x)), {"x": handle}) == "f((x))"  # an object has no literal
assert text(b("+", x, y), {"x": 1.5}) == "1.5 + y" and text(b("+", y, b("*", 1.5, 2)), SVV.Scope(types={"y": REAL})) == "y + 3.0"
shortreal = lambda operand: SV.cast(lambda t: t.nonintegertype(lambda n: n.keyword("shortreal")), operand)  # noqa: E731
assert text(b("+", shortreal(0.1), y), SVV.Scope(types={"y": SHORTREAL})) == "shortreal'(0.1) + y"  # no literal is one
assert text(b("+", shortreal(0.1), y), SVV.Scope(types={"y": REAL})) == "0.10000000149011612 + y"
# `**`'s exponent is self-determined where the power is integral, its context's where it is real.
assert text(b("**", y, b("+", 1, 1))) == "y ** (1 + 1)" and text(b("**", y, b("+", 1, 1)), SVV.Scope(types={"y": INTEGER})) == "y ** 2"
assert text(b("**", y, b("+", 1, 1)), SVV.Scope(types={"y": REAL})) == "y ** 2.0" and text(SV.replication(b("+", 1, 1), y)) == "{2{y}}"
# A name is its value's literal, of its own type, wherever it stands.
assert text(b("+", x, y), {"x": SVD.Logic.of("1x")}) == "2'b1x + y" and text(b("+", x, y), {"x": 5}) == "5 + y"
# `&&` and `||` are decided by an operand that decides them, `->` by a true consequent (Kleene's logic, x unknown).
assert text(b("&&", y, 0)) == "1'b0" and text(b("||", y, v("1"))) == "1'b1" and text(b("->", y, 1)) == "1'b1"
assert text(b("&&", y, 1)) == "y && 1" and text(b("||", 0, y)) == "0 || y"
# A conditional by a known condition, where the branch it keeps has the conditional's own type.
assert text(SV.conditional(1, y, b("+", 1, 1)), SVV.Scope(types={"y": INTEGER})) == "y"
assert text(SV.conditional(0, v("1111"), y), SVV.Scope(types={"y": INTEGER})) == "0 ? 4'b1111 : y"
assert text(SV.conditional(1, y, v("1111")), SVV.Scope(types={"y": INTEGER})) == "1 ? y : 4'b1111"
assert text(SV.conditional(0, b("+", 1, 1), y), SVV.Scope(types={"y": INTEGER})) == "y" and text(SV.conditional(1, y, 2)) == "1 ? y : 2"
assert text(SV.conditional(v("x"), y, 1), SVV.Scope(types={"y": INTEGER})) == "1'bx ? y : 1"
# What a scope does not provide stays, as it was: a name, a function, and what they size.
unbound = simplified(b("+", x, call("f", 1)))
assert text(b("+", x, call("f", 1))) == "x + f(1)" and unbound.free == {"x", "f"} and unbound.residues == ()
with raises(NotEvaluated, match="the root is not a closed constructor expression: f is not bound"):
    unbound.evaluate()
# What IEEE 1800 may never evaluate stays, though it would raise: a branch, by an unknown condition; `&&`'s right operand
known, unknown, struct = b("==", SV.identifier("y"), 1), b("==", SV.identifier("x"), 1), {"y": {"a": 1}}
assert text(SV.conditional(unknown, 1, known), struct) == "x == 1 ? 1 : y == 1" and text(b("&&", unknown, known), struct) == "x == 1 && y == 1"
with raises(TypeError, match="== expects a number, got dict"):  # an operand it evaluates raises
    text(b("+", unknown, known), struct)

<a id="SVS-03"></a>
## SVS-03 · Casts: an integral type's operand sized as an assignment, vector types of constant dimensions, signings, `realtime` and `string`

In [ ]:
# A cast to an integral type takes its operand as an assignment would (6.24.1): computed in the wider width, by the
# operand's own signedness, then converted.
assert value(SV.cast(16, b("+", v("11111111"), v("00000001")))) == "16'b0000000100000000"
assert value(SV.cast(atom("int"), b("+", v("11111111"), v("00000001")))) == integer(256)
assert value(SV.cast(4, b("+", v("11111111"), v("00000001")))) == "4'b0000"
assert value(SV.cast(lambda t: t.implicittype(lambda i: i.signing("signed")), b("+", v("1111"), v("0001")))) == "4'sb0000"
# Vector types of constant dimensions, signed or not, a keyword's type with a signing, `realtime` (`real`), `string`.
assert value(SV.cast(vector_type("logic", 7, 0, "signed"), v("1111"))) == "8'sb00001111"
assert value(SV.cast(vector_type("bit", 0, 3), v("1x01"))) == "4'b1001"  # 2-state: x is 0
assert value(SV.cast(lambda t: t.integervectortype(lambda n: n.keyword("logic").dimensions([
    lambda d: d.rangedimension(lambda r: r.left(b("-", 4, 1)).right(SV.literal(0))),
    lambda d: d.rangedimension(lambda r: r.left(SV.literal(1)).right(SV.literal(0)))])), -1)) == "8'b11111111"
assert value(SV.cast(atom("int", "unsigned"), -1)) == integer(-1, 32, False) and value(SV.cast(atom("byte", "unsigned"), 300)) == "8'b00101100"
assert value(SV.cast(lambda t: t.nonintegertype(lambda n: n.keyword("realtime")), 2)) == 2.0
string_ = lambda t: t.keywordtype(lambda k: k.keyword("string"))  # noqa: E731
assert value(SV.cast(string_, v("0000000001000001"))) == "8'b01000001"  # its bytes, NUL bytes dropped
with raises(TypeError, match="string' expects an integral operand, got float"):
    value(SV.cast(string_, 1.5))
# The type's terms fold where they stand: a dimension's bound is self-determined.
assert text(SV.cast(lambda t: t.integervectortype(lambda n: n.keyword("logic").dimensions([
    lambda d: d.rangedimension(lambda r: r.left(b("-", 8, 1)).right(SV.literal(0)))])), y)) == "logic [7:0]'(y)"
# A cast to a type with no value read here stays, named: a structure's, an enumeration's (domain inference, c), a
# typedef's name (its declaration, d), an unpacked dimension (b).
named = lambda t: t.namedtype(lambda n: n.name(lambda i: i.identifier(lambda d: d.spelling("word"))))  # noqa: E731
unpacked = lambda t: t.integervectortype(lambda n: n.keyword("logic").dimensions([  # noqa: E731
    lambda d: d.sizedimension(lambda r: r.size(SV.literal(8)))]))
assert reasons(SV.cast(lambda t: t.structtype(lambda r: r.keyword("struct")), 1)) == [
    ("SystemVerilog castexpression to structtype is not simplified", "c")]
assert reasons(SV.cast(named, 1)) == [("SystemVerilog castexpression to namedtype is not simplified", "d")]
assert reasons(SV.cast(unpacked, 1)) == [("SystemVerilog Types sizedimension is not simplified", "b")]

<a id="SVS-04"></a>
## SVS-04 · Selects: a queue's last index `$`, slices of arrays and queues, and indexed part-selects of vectors and arrays

In [ ]:
# `$` is the last index of the array or the queue its select indexes (7.10.1).
assert value(SV.select(q, dollar), {"q": [1, 2, 3]}) == integer(3) and value(SV.select(q, b("-", dollar, 1)), {"q": [1, 2, 3]}) == integer(2)
assert rule(SV.range_(q, 1, dollar), {"q": [1, 2, 3]}) == [integer(2), integer(3)]
assert rule(SV.range_(q, 2, 1), {"q": [1, 2, 3]}) == [] and rule(SV.range_(q, -1, 7), {"q": [1, 2]}) == [integer(1), integer(2)]
assert text(SV.select(q, dollar)) == "q[$]" and text(SV.select(y, b("-", dollar, b("+", 1, 1)))) == "y[$ - 2]"
with raises(TypeError, match="$ is the last index of a queue or an array, not of a vector"):
    value(SV.select(v("1010"), dollar))
# An indexed part-select (11.5.1): `[base +: width]`, bits from base up, `[base -: width]`, bits from base down, its
# width a constant; out of range x, and at an x base all x. Of an array or a queue, its items.
part = lambda target, base, operator, width: lambda u_: u_.rangeselect(  # noqa: E731
    lambda r: r.value(target).left(base).operator(operator).right(SV.literal(width)))
assert value(part(v("11001010"), SV.literal(2), "+:", 4)) == "4'b0010" and value(part(v("11001010"), SV.literal(5), "-:", 3)) == "3'b001"
assert value(part(v("1010"), SV.literal(2), "+:", 4)) == "4'bxx10" and value(part(v("1010"), v("x"), "+:", 2)) == "2'bxx"
assert rule(part(q, SV.literal(1), "+:", 2), {"q": [5, 6, 7]}) == [integer(6), integer(7)]
assert rule(part(q, dollar, "-:", 2), {"q": [5, 6, 7]}) == [integer(6), integer(7)]
assert text(part(y, b("+", 1, 1), "+:", 4)) == "y[2 +: 4]"
for spec, error, message in [
    (part(v("1010"), SV.literal(0), "+:", 0), ValueError, "a part-select's width must be positive, got 0"),
    (part(q, v("x"), "+:", 1), ValueError, "a part-select's base must be known, got 1'bx"),
    (SV.range_(SV.literal(1.5), 1, 0), TypeError, "a float cannot be selected from"),
]:
    with raises(error, match=message):
        value(spec, {"q": [1]})

<a id="SVS-05"></a>
## SVS-05 · `inside` with tolerance ranges `[a +/- b]` and `[a +%- b]`, and a range bounded by its type's `$`

In [ ]:
tolerance = lambda a, operator, b_: lambda u_: u_.valuerange(  # noqa: E731
    lambda r: r.left(SV.literal(a)).operator(operator).right(SV.literal(b_)))
assert [value(SV.inside(n, tolerance(10, "+/-", 2))) for n in (7, 8, 12, 13)] == ["1'b0", "1'b1", "1'b1", "1'b0"]
assert [value(SV.inside(n, tolerance(100, "+%-", 10))) for n in (89, 90, 110, 111)] == ["1'b0", "1'b1", "1'b1", "1'b0"]
assert value(SV.inside(10.5, tolerance(10.0, "+%-", 5.0))) == "1'b1" and value(SV.inside(v("1x00"), tolerance(12, "+/-", 1))) == "1'bx"
# `$` bounds a range by its type's least or greatest value (11.4.13).
assert value(SV.inside(v("11111111"), SV.span(v("00000010"), dollar))) == "1'b1"
assert value(SV.inside(-5, SV.span(dollar, 0))) == "1'b1" and value(SV.inside(1.0e300, SV.span(0.0, dollar))) == "1'b1"
assert value(SV.inside(-1.0e300, SV.span(dollar, 0.0))) == "1'b1"
assert text(SV.inside(y, SV.span(b("+", 1, 1), dollar))) == "y inside {[1 + 1:$]}"  # its type is `y`'s too

<a id="SVS-06"></a>
## SVS-06 · `matches` and `&&&`: constant, wildcard and variable patterns, a variable bound in its conditional's consequence

In [ ]:
def matches(value_, pattern):
    return lambda u_: u_.matchesexpression(lambda m: m.value(value_).pattern(pattern))


variable = lambda name: lambda u_: u_.variablepattern(lambda p_: p_.name(lambda i: i.spelling(name)))  # noqa: E731
wildcard = lambda u_: u_.wildcardpattern(lambda w: w)  # noqa: E731
predicate = lambda *conditions: lambda u_: u_.predicateexpression(  # noqa: E731
    lambda r: r.conditions([SV.DIALECT.spec(condition) for condition in conditions]))
a = SV.identifier("a")
# A constant pattern matches a value identical to it (`===`); a wildcard and a variable any value (12.6).
assert [value(matches(x, SV.literal(5)), {"x": n}) for n in (5, 6)] == ["1'b1", "1'b0"]
assert value(matches(x, v("1x")), {"x": SVD.Logic.of("1x")}) == "1'b1" and value(matches(x, wildcard), {"x": 1}) == "1'b1"
assert value(matches(x, lambda u_: u_.parenthesizedpattern(lambda r: r.pattern(SV.literal(5)))), {"x": 5}) == "1'b1"
# A variable pattern binds the value in its conditional's consequence, and in the conditions after it in a `&&&`.
assert value(SV.conditional(matches(x, variable("a")), b("+", a, 1), 0), {"x": 41}) == integer(42)
guarded = SV.conditional(predicate(matches(x, variable("a")), b(">", a, 2)), a, 0)
assert value(guarded, {"x": 9}) == integer(9) and value(guarded, {"x": 1}) == integer(0)
assert value(predicate(b(">", x, 0), b("<", x, 9)), {"x": 3}) == "1'b1" and value(predicate(v("x"), 1)) == "1'bx"
assert value(predicate(0, v("x"))) == "1'b0"
assert value(SV.conditional(predicate(matches(x, SV.literal(5)), matches(x, variable("a"))), a, 0), {"x": 5}) == integer(5)
# What it binds stays bound: a `matches` with a variable stays as written, its consequence's names its own.
assert text(SV.conditional(matches(y, variable("a")), b("+", a, b("+", 1, 2)), 0)) == "y matches .a ? a + (1 + 2) : 0"
assert text(SV.conditional(matches(y, SV.literal(5)), b("+", 1, 2), 0)) == "y matches 5 ? 3 : 0"
assert text(matches(y, lambda u_: u_.parenthesizedpattern(lambda r: r.pattern(b("+", 2, 3))))) == "y matches (2 + 3)"
assert text(SV.conditional(predicate(b(">", y, 0), matches(x, variable("a")), b(">", a, b("+", 1, 1))), a, 0), {"x": 5}) == (
    "y > 0 &&& 5 matches .a &&& a > 2 ? a : 0")
assert text(predicate(matches(y, variable("a")), b(">", a, b("+", 1, 1)))) == "y matches .a &&& a > 1 + 1"
# A tagged or a structure pattern waits on its tagged union's or its structure's type (c).
tagged = lambda u_: u_.taggedpattern(lambda t: t.member(lambda i: i.spelling("Valid")))  # noqa: E731
structure = lambda u_: u_.structurepattern(lambda r: r.items([wildcard]))  # noqa: E731
assert reasons(SV.conditional(matches(x, tagged), 1, 0), {"x": 1}) == [
    ("SystemVerilog matchesexpression taggedpattern is not simplified", "c")]
assert reasons(matches(x, structure), {"x": 1}) == [("SystemVerilog matchesexpression structurepattern is not simplified", "c")]

<a id="SVS-07"></a>
## SVS-07 · Streaming concatenations, `min:typ:max` by the scope's choice, and the empty queue

In [ ]:
def stream(operator, slice_, *items):
    """`{<< slice {items}}`, each item an expression, or a pair of an array and its `with` range."""
    def item(it):
        if isinstance(it, tuple):
            array, (left, operator_, right) = it
            return lambda i: i.expression(array).left(SV.literal(left)).operator(operator_).right(SV.literal(right))
        return lambda i: i.expression(SV.DIALECT.spec(it))
    return lambda u_: u_.streamingconcatenation(
        lambda r: (r.operator(operator) if slice_ is None else r.operator(operator).slice(slice_)).items([item(it) for it in items]))


# `>>` streams the items' bits as they stand; `<<` takes slices from the right and places them left to right (11.4.14).
assert value(stream("<<", SV.literal(4), v("110101"))) == "6'b010111" and value(stream(">>", SV.literal(4), v("110101"))) == "6'b110101"
assert value(stream("<<", None, v("00110101"))) == "8'b10101100"  # a slice of 1 bit: the bits reversed
assert value(stream("<<", atom("byte"), v("1010101011110000"))) == "16'b1111000010101010"  # a type's width
assert value(stream("<<", SV.literal(2), stream("<<", None, v("1101")))) == "4'b1110"
assert value(stream(">>", None, q, v("1")), {"q": [SVD.Logic.of("10"), SVD.Logic.of("0x")]}) == "5'b100x1"  # an array's items
assert value(stream(">>", None, (q, (1, ":", 2))), {"q": [SVD.Logic.of("1"), SVD.Logic.of("0"), SVD.Logic.of("1")]}) == "2'b01"
assert value(stream(">>", None, (q, (0, "+:", 2))), {"q": [SVD.Logic.of("1"), SVD.Logic.of("0"), SVD.Logic.of("1")]}) == "2'b10"
assert value(stream(">>", None, (q, (2, "-:", 2))), {"q": [SVD.Logic.of("1"), SVD.Logic.of("0"), SVD.Logic.of("1")]}) == "2'b01"
assert text(stream("<<", b("+", 2, 2), y)) == "{<< 4 {y}}" and text(stream("<<", None, b("+", y, 1))) == "{<<{y + 1}}"
assert text(stream("<<", atom("byte"), y, b("+", v("01"), v("01")))) == "{<< byte {y, 2'b10}}"
for spec, error, message in [(stream("<<", SV.literal(0), v("1")), ValueError, "a stream's slice must be positive, got 0"),
                             (stream(">>", None, (v("1"), (0, ":", 0))), TypeError, "a stream's `with` selects from an array, not a Logic"),
                             (stream(">>", None, 1.5), TypeError, "a concatenation's parts must be integral, got float")]:
    with raises(error, match=message):
        value(spec)
# `min:typ:max` is the expression the scope chooses (11.11), `typ` by default.
choice = lambda u_: u_.mintypmaxexpression(lambda r: r.min(SV.literal(1)).typ(b("+", 1, 1)).max(SV.literal(3)))  # noqa: E731
assert [value(choice, SVV.Scope(mintypmax=chosen)) for chosen in ("min", "typ", "max")] == [integer(1), integer(2), integer(3)]
assert value(b("+", v("0001"), choice)) == integer(3, 32, False)  # unsigned, as `4'b0001` is
assert text(lambda u_: u_.mintypmaxexpression(lambda r: r.min(SV.literal(1)).typ(y).max(b("+", 1, 2)))) == "1:y:1 + 2"
# The empty queue is an empty array, which no literal writes but `{}`.
empty = lambda u_: u_.emptyqueue(lambda e: e)  # noqa: E731
assert rule(empty) == [] and text(empty) == "{}" and value(SV.method(empty, "size")) == integer(0, 32)

<a id="SVS-08"></a>
## SVS-08 · Calls of the scope's functions: by position, by name and with defaults (`Subroutine`); a class handle's methods stay

In [ ]:
three = SVV.Subroutine(lambda a_, b_, c_: 100 * a_.integer() + 10 * b_.integer() + c_.integer(), ("a", "b", "c"), {"b": 5, "c": 6})
scope = SVV.Scope({"c": 7}, {"f": three, "g": lambda n: n})


def named(name, value_=None, implicit=False):
    """`.name(value)`, `.name()` or `.name`."""
    def write(c):
        c.name(lambda i: i.spelling(name))
        return c.value(SV.DIALECT.spec(value_)) if value_ is not None else c.implicit(True) if implicit else c
    return lambda u_: u_.namedconnection(write)


empty_ = lambda u_: u_.emptyargument(lambda e: e)  # noqa: E731
f = lambda function, *arguments: lambda u_: u_.callexpression(  # noqa: E731
    lambda r: r.callee(SV.identifier(function)).arguments([SV.DIALECT.spec(a_) for a_ in arguments]))
assert value(f("f", 1, 2, 3), scope) == integer(123) and value(f("f", 1), scope) == integer(156)
assert value(f("f", 1, empty_, 3), scope) == integer(153) and value(f("f", 1, named("c", 9)), scope) == integer(159)
assert value(f("f", named("a", 2), named("b")), scope) == integer(256) and value(f("f", 1, named("c", implicit=True)), scope) == integer(157)
assert text(f("f", b("+", 1, 1), named("c", b("+", y, 1))), scope) == "f(2, .c(y + 1))"
for spec, message in [(f("f"), "f's argument 'a' has no default"), (f("f", 1, 2, 3, 4), "f takes 3 arguments, its positional ones first"),
                      (f("f", named("a", 1), 2), "f takes 3 arguments, its positional ones first"),
                      (f("f", named("d", 1)), "f has no formal 'd', or connects it twice"),
                      (f("f", named("a", 1), named("a", 2)), "f has no formal 'a', or connects it twice")]:
    with raises(TypeError, match=message):
        value(spec, scope)
# A function the scope gives as a callable has no formals: named or empty arguments stay (a), as the call does.
assert reasons(f("g", named("n", 1)), scope) == [("SystemVerilog callexpression with a namedconnection is not simplified", "a")]
assert reasons(f("g", empty_), scope) == [("SystemVerilog callexpression with an emptyargument is not simplified", "a")]
# Alone, a named connection or an empty argument has no value (b); a class handle's method needs its class (d).
assert reasons(named("a", 1)) == [("SystemVerilog namedconnection is not simplified", "b")]
assert reasons(SV.method(x, "start"), {"x": handle}) == [("SystemVerilog callexpression of a class handle's method is not simplified", "d")]
local = lambda u_: u_.nameexpression(lambda r: r.name(lambda n: n.localname(lambda c: c.name(lambda i: i.spelling("f")))))  # noqa: E731
assert reasons(lambda u_: u_.callexpression(lambda r: r.callee(local).arguments([]))) == [
    ("SystemVerilog callexpression of a localname is not simplified", "d")]
assert reasons(lambda u_: u_.callexpression(lambda r: r.callee(parenthesized(SV.identifier("g"))).arguments([])), scope) == [
    ("SystemVerilog callexpression of a parenthesizedexpression is not simplified", "a")]

<a id="SVS-09"></a>
## SVS-09 · Strings' methods (6.16): `len`, `getc`, `toupper`, `tolower`, `compare`, `icompare`, `substr` and the conversions

In [ ]:
word = {"s": "aBc"}
assert value(SV.method(s, "len"), word) == integer(3) and value(SV.method(SV.literal("hello"), "len")) == integer(5)
assert value(SV.method(s, "getc", 1), word) == "8'sb01000010" and value(SV.method(s, "getc", 9), word) == "8'sb00000000"
assert value(SV.method(s, "toupper"), word) == value(SV.literal("ABC")) and value(SV.method(s, "tolower"), word) == value(SV.literal("abc"))
assert value(SV.method(s, "substr", 1, 2), word) == value(SV.literal("Bc")) and value(SV.method(s, "substr", 2, 1), word) == "8'b00000000"
assert value(SV.method(s, "compare", "abd"), word) == integer(-1) and value(SV.method(s, "icompare", "ABC"), word) == integer(0)
assert value(SV.method(s, "compare", "aBc"), word) == integer(0) and value(SV.method(s, "compare", "a"), word) == integer(1)
for method, text_, found in [("atoi", "-1_234x", integer(-1234)), ("atoi", "+12", integer(12)), ("atoi", "x1", integer(0)),
                             ("atohex", "fF_g", integer(255)), ("atooct", "778", integer(63)), ("atobin", "1012", integer(5))]:
    assert value(SV.method(s, method), {"s": text_}) == found, (method, text_)
assert [value(SV.method(s, "atoreal"), {"s": t}) for t in ("1_2.5e1z", "-.5", ".e3", "")] == [125.0, -0.5, 0.0, 0.0]
assert Formats.text(simplified(SV.method(s, "toupper"), {"s": "é"}).evaluate()) == "\xc3\xa9"  # bytes, ASCII's letters alone change
for spec, error, message in [(SV.method(s, "getc"), TypeError, "getc() takes 1 arguments, got 0"),
                             (SV.method(1.5, "len"), TypeError, "len() is a method of strings, not of float"),
                             (SV.method(s, "compare", 1.5), TypeError, "compare() compares strings, not a float"),
                             (SV.method(s, "getc", v("x")), ValueError, "getc's index must be a known integer, got 1'bx")]:
    with raises(error, match=message):
        value(spec, word)
# The methods that write the string are a statement's (d).
assert reasons(SV.method(s, "putc", 0, 65), word) == [("SystemVerilog callexpression method 'putc' is not simplified", "d")]

<a id="SVS-10"></a>
## SVS-10 · Arrays' locator methods (7.12.1), with or without `with`, and an item's `index()`; a method that writes stays

In [ ]:
digits = {"q": [3, 1, 4, 1, 5]}
odd = b("%", p, 2)
locate = lambda method, body: SV.iterate(q, method, "p", body)  # noqa: E731
assert rule(locate("find", b(">", p, 1)), digits) == [integer(3), integer(4), integer(5)]
assert rule(locate("find_index", b(">", p, 1)), digits) == [integer(0), integer(2), integer(4)]
assert rule(locate("find_first", b(">", p, 3)), digits) == [integer(4)] and rule(locate("find_first_index", b(">", p, 9)), digits) == []
assert rule(locate("find_last", b("<", p, 4)), digits) == [integer(1)] and rule(locate("find_last_index", b("<", p, 4)), digits) == [integer(3)]
assert rule(locate("min", odd), digits) == [integer(4)] and rule(locate("max", odd), digits) == [integer(3)]
assert rule(locate("unique", odd), digits) == [integer(3), integer(4)] and rule(locate("unique_index", odd), digits) == [integer(0), integer(2)]
assert rule(SV.method(q, "unique_index"), digits) == [integer(0), integer(1), integer(2), integer(4)]
assert value(SV.method(locate("find", b(">", p, 1)), "size"), digits) == integer(3)  # a queue: its size folds
item_index = lambda u_: u_.callexpression(lambda r: r.callee(SV.member(p, "index")).arguments([]))  # noqa: E731
assert rule(locate("find", b("==", item_index, 1)), {"q": [7, 8, 9]}) == [integer(8)]
assert value(locate("sum", b("*", p, item_index)), {"q": [7, 8, 9]}) == integer(26)
assert text(locate("find", b(">", p, b("+", 1, 1)))) == "q.find(p) with (p > 1 + 1)"
not_a_method = lambda u_: u_.arraymethodwithexpression(lambda r: r.value(call("f", p)).body(b("+", 1, 2)))  # noqa: E731
assert text(not_a_method) == "f(p) with (3)" and reasons(not_a_method) == [
    ("SystemVerilog arraymethodwithexpression of a call that is not an array method's is not simplified", "a")]
# A method that writes the array is a statement's (d); any other the rules do not fold stays (a).
assert reasons(SV.method(q, "push_back", 1), digits) == [("SystemVerilog callexpression method 'push_back' is not simplified", "d")]
assert reasons(SV.method(q, "sum", 1), digits) == [("SystemVerilog callexpression method 'sum' is not simplified", "a")]
assert reasons(SV.method(x, "next"), {"x": 1}) == [("SystemVerilog callexpression method 'next' is not simplified", "c")]

<a id="SVS-11"></a>
## SVS-11 · System functions: the math functions (C's library), the conversions between reals and bits, `$countbits`, `$isunbounded`

In [ ]:
# The math functions take reals, an integral argument converted, and give reals as C's library does: a domain error NaN,
# a pole or an overflow an infinity. Exact where IEEE 754 makes them so; the others are each host's library's.
assert [value(call(name, 2)) for name in ("$sqrt", "$floor", "$ceil")] == [math.sqrt(2.0), 2.0, 2.0]
assert value(call("$pow", 2, 10)) == 1024.0 and value(call("$floor", -1.5)) == -2.0 and value(call("$ceil", 2.5)) == 3.0
assert math.copysign(1.0, value(call("$ceil", -0.5))) == -1.0 and value(call("$floor", math.inf)) == math.inf
assert [value(call(name, 0)) for name in ("$ln", "$log10")] == [-math.inf, -math.inf] and value(call("$atanh", 1)) == math.inf
assert all(math.isnan(value(call(name, -1))) for name in ("$ln", "$log10", "$sqrt")) and math.isnan(value(call("$asin", 2)))
assert math.isnan(value(call("$acosh", 0.5))) and math.isnan(value(call("$atanh", 2))) and value(call("$atanh", -1)) == -math.inf
assert value(call("$exp", 1000)) == math.inf and value(call("$sinh", -1000)) == -math.inf and value(call("$cosh", 1000)) == math.inf
assert [value(call(name, 0)) for name in ("$exp", "$sin", "$cos", "$tan", "$asin", "$atan", "$sinh", "$cosh", "$tanh", "$asinh")] == [
    1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0]
assert value(call("$acos", 1)) == 0.0 and value(call("$acosh", 1)) == 0.0 and value(call("$atan2", 0, 1)) == 0.0
assert value(call("$hypot", 3, 4)) == 5.0 and value(call("$log10", 1000)) == 3.0 and value(call("$ln", 1)) == 0.0
with raises(TypeError, match="$sqrt expects a number, got dict"):
    value(call("$sqrt", x), {"x": {}})
with raises(TypeError, match="$pow takes 2 arguments, got 1"):
    value(call("$pow", 2))
# The conversions (20.5): `$rtoi` truncates to an `integer`, `$itor` converts one to a real; a real's encoding and back.
assert value(call("$rtoi", -2.7)) == integer(-2) and value(call("$rtoi", 3)) == integer(3) and value(call("$itor", v("11111111"))) == 255.0
assert value(call("$itor", 2.5)) == 3.0  # assigned to an integer first: rounded
assert value(call("$realtobits", 1.0)) == "64'b0011111111110000" + "0" * 48 and value(call("$bitstoreal", call("$realtobits", -1.5))) == -1.5
assert value(call("$shortrealtobits", 1.0)) == "32'b00111111100000000000000000000000"
assert value(call("$bitstoshortreal", call("$shortrealtobits", 0.1))) == 0.10000000149011612
assert value(b("+", call("$bitstoshortreal", v("00111111100000000000000000000000")), SV.cast(lambda t: t.nonintegertype(
    lambda n: n.keyword("shortreal")), 0.1))) == 1.100000023841858  # a shortreal's sum, in binary32
with raises(OverflowError, match="nan has no integer value"):
    value(call("$rtoi", math.nan))
with raises(TypeError, match="$bitstoreal expects an integral argument, got real"):
    value(call("$bitstoreal", 1.5))
# `$countbits` counts the bits in the states its controls name; `$isunbounded` tells `$`.
assert value(call("$countbits", v("10xz10"), fill("1"), fill("x"))) == integer(3) and value(call("$countbits", v("10xz10"), v("0"))) == integer(2)
assert value(call("$isunbounded", dollar)) == "1'b1" and value(call("$isunbounded", 1)) == "1'b0"
with raises(TypeError, match="$countbits takes 2 arguments or more, got 1"):
    value(call("$countbits", 1))
with raises(TypeError, match="$countbits expects integral arguments, got float"):
    value(call("$countbits", 1, 1.5))

<a id="SVS-12"></a>
## SVS-12 · Array queries (20.7) of a value or a type: `$left`, `$right`, `$low`, `$high`, `$increment`, `$size`, `$dimensions`, `$unpacked_dimensions`

In [ ]:
QUERIES = ["$left", "$right", "$low", "$high", "$increment", "$size", "$dimensions", "$unpacked_dimensions"]
nested = {"x": [[1, 2], [3, 4], [5, 6]]}
assert [value(call(name, x), nested) for name in QUERIES] == [integer(n) for n in (0, 2, 0, 2, -1, 3, 3, 2)]
assert [value(call(name, x, 2), nested) for name in QUERIES[:6]] == [integer(n) for n in (0, 1, 0, 1, -1, 2)]
assert [value(call(name, x, 3), nested) for name in QUERIES[:6]] == [integer(n) for n in (31, 0, 0, 31, 1, 32)]
assert value(call("$left", x, 4), nested) == integer(-1).replace("1", "x") and value(call("$size", x), {"x": []}) == integer(0)
assert [value(call(name, x), {"x": True}) for name in ("$dimensions", "$left")] == [integer(0), "32'sb" + "x" * 32]  # a scalar
assert [value(call(name, vector_type("logic", 0, 7))) for name in QUERIES] == [integer(n) for n in (0, 7, 0, 7, -1, 8, 1, 0)]
assert [value(call(name, atom("byte"))) for name in ("$left", "$size", "$dimensions")] == [integer(7), integer(8), integer(1)]
assert value(call("$dimensions", lambda t: t.nonintegertype(lambda n: n.keyword("real")))) == integer(0)
assert value(call("$dimensions", lambda t: t.integervectortype(lambda n: n.keyword("logic")))) == integer(0)  # a scalar
assert value(call("$size", vector_type("logic", 7, 0))) == integer(8)
with raises(TypeError, match="$left takes 1 or 2 arguments, got 3"):
    value(call("$left", x, 1, 1), nested)
assert reasons(call("$left", lambda t: t.integervectortype(lambda n: n.keyword("logic").dimensions([
    lambda d: d.unsizeddimension(lambda r: r)])))) == [("SystemVerilog Types unsizeddimension is not simplified", "b")]

<a id="SVS-13"></a>
## SVS-13 · `$sformatf` writes its arguments as its format says (21.2.1), reals as C writes them

In [ ]:
def written(format_, *arguments, scope=None):
    return Formats.text(simplified(call("$sformatf", format_, *arguments), scope).evaluate())


assert written("%d|%0d|%5d|%-5d|%05d", 5, 5, 5, 5, 5) == "          5|5|    5|5    |00005"
assert written("%d %d %0d", v("10x1"), v("xxxx"), SV.cast(8, -1)) == " X  x -1" and written("%d", v("zzzz")) == " z"
assert written("%d", v("10z1")) == " Z" and written("%4h", v("0000")) == "0000"
assert written("%h %0h %4h %b %o %x", v("00101x01"), v("00101101"), v("00101101"), v("101"), v("111000"), v("zzzz1z1z")) == "2X 2d 002d 101 70 zZ"
assert written("%h %h", v("xxxx"), v("zzzz")) == "x z" and written("%s=%c%%", "ab", 65) == "ab=A%" and written("%5s|%-3s|", "ab", "c") == "   ab|c  |"
assert written("%e %f %g %.2f %10.3e %G", 1.5, 1.5, 0.0001, 2.675, 12345.678, 1e-10) == (
    "1.500000e+00 1.500000 0.0001 2.67  1.235e+04 1e-10")
assert written("%f %e %g %08.2f %-8.1f|", math.inf, -math.inf, math.nan, -3.14159, 2.0) == "inf -inf nan -0003.14 2.0     |"
assert written("%g %g %g %.3g", 100000.0, 1000000.0, 0.00001, 2.5) == "100000 1e+06 1e-05 2.5" and written("%f", 7) == "7.000000"
assert written("no specification") == "no specification" and written("%0d%%%0d", 1, 2) == "1%2"
# Formats.real writes a real as C does, from its exact value, rounded half to even: `%e`, `%.0e`, `%.3e`, then `%f`
# and `%g` alike; and Python's own `%`, which is C's, says so of more.
for number, written_ in [
    (0.0, '0.000000e+00 0e+00 0.000e+00 0.000000 0 0.000 0 0 0'),
    (-0.0, '-0.000000e+00 -0e+00 -0.000e+00 -0.000000 -0 -0.000 -0 -0 -0'),
    (0.5, '5.000000e-01 5e-01 5.000e-01 0.500000 0 0.500 0.5 0.5 0.5'),
    (2.5, '2.500000e+00 2e+00 2.500e+00 2.500000 2 2.500 2.5 2 2.5'),
    (0.1, '1.000000e-01 1e-01 1.000e-01 0.100000 0 0.100 0.1 0.1 0.1'),
    (1e-05, '1.000000e-05 1e-05 1.000e-05 0.000010 0 0.000 1e-05 1e-05 1e-05'),
    (123456.789, '1.234568e+05 1e+05 1.235e+05 123456.789000 123457 123456.789 123457 1e+05 1.23e+05'),
    (5e-324, '4.940656e-324 5e-324 4.941e-324 0.000000 0 0.000 4.94066e-324 5e-324 4.94e-324'),
    (9.9999995, '9.999999e+00 1e+01 1.000e+01 9.999999 10 10.000 10 1e+01 10'),
    (1e+16, '1.000000e+16 1e+16 1.000e+16 10000000000000000.000000 10000000000000000 10000000000000000.000 1e+16 1e+16 1e+16'),
]:
    assert " ".join(Formats.real(c, number, p_) for c in "efg" for p_ in (None, 0, 3)) == written_, number
for number in [0.0, -0.0, 0.5, 1.5, 2.5, 0.1, 1e-5, 123456.789, 1e300, 5e-324, 9.9999995, 99999.95, 0.00012345, 1e16]:
    for conversion in "efg":
        for precision in (None, 0, 1, 3, 10):
            expected = ("%" + ("" if precision is None else f".{precision}") + conversion) % number
            assert Formats.real(conversion, number, precision) == expected, (number, conversion, precision)
for spec, error, message in [(call("$sformatf", "%d"), TypeError, "$sformatf's format has more specifications than arguments"),
                             (call("$sformatf", "", 1), TypeError, "$sformatf's format has fewer specifications than arguments"),
                             (call("$sformatf", "%q", 1), ValueError, "%q is not a format specification"),
                             (call("$sformatf", "%d", 1.5), TypeError, "%d expects an integral argument, got real"),
                             (call("$sformatf", 1.5), TypeError, "$sformatf's format is a string, not a float"),
                             (call("$sformatf", "%d", x), TypeError, "$sformatf writes numbers and strings, not a dict"),
                             (call("$sformatf"), TypeError, "$sformatf takes 1 argument or more, got 0")]:
    with raises(error, match=message):
        simplified(spec, {"x": {}})
# What reads a simulator's settings or hierarchy stays (d).
assert reasons(call("$sformatf", "%t", 1)) == [("SystemVerilog systemcall '$sformatf' %t is not simplified", "d")]

<a id="SVS-14"></a>
## SVS-14 · What stays, by its class: `FOLDED`, `CATEGORIES`, and each kind no rule folds named in the residual

In [ ]:
assert len(SVV.FOLDED) == 26 and {"dollarexpression", "streamingconcatenation", "mintypmaxexpression", "emptyqueue",
                                  "namedconnection", "emptyargument", "matchesexpression", "predicateexpression",
                                  "thisexpression"} <= SVV.FOLDED
assert set(SV.TAGS) - SVV.FOLDED == {"incrementexpression", "assignmentexpression", "valuerange", "assignmentpattern",
                                     "patternitem", "superexpression", "newexpression", "newcopyexpression",
                                     "newarrayexpression", "randomizewithexpression", "taggedexpression", "rootexpression"}
for spec, found in [
    (lambda u_: u_.incrementexpression(lambda r: r.operator("++").operand(x)), ("SystemVerilog incrementexpression is not simplified", "d")),
    (lambda u_: u_.assignmentpattern(lambda r: r.items([SV.literal(1)])), ("SystemVerilog assignmentpattern is not simplified", "c")),
    (lambda u_: u_.taggedexpression(lambda r: r.member(lambda i: i.spelling("Valid"))), ("SystemVerilog taggedexpression is not simplified", "c")),
    (lambda u_: u_.rootexpression(lambda r: r), ("SystemVerilog rootexpression is not simplified", "d")),
    (SV.span(1, 2), ("SystemVerilog valuerange is not simplified", "b")),
    (call("$time"), ("SystemVerilog systemcall '$time' is not simplified", "d")),
    (call("$typename", 1), ("SystemVerilog systemcall '$typename' is not simplified", "c")),
    (lambda u_: u_.nameexpression(lambda r: r.name(lambda n: n.localname(lambda c: c.name(lambda i: i.spelling("x"))))),
     ("SystemVerilog nameexpression of a localname is not simplified", "d")),
    (call("$bits", lambda t: t.keywordtype(lambda k: k.keyword("string"))), ("SystemVerilog systemcall '$bits' of keywordtype is not simplified", "b")),
]:
    assert reasons(spec, {"x": 1}) == [found], (found, reasons(spec, {"x": 1}))
# Within a sublanguage's term, which no rule reads, an operand's context is the sublanguage's: what passes it on stays.
first_match = lambda inner: lambda u_: u_.firstmatchsequence(lambda r: r.sequence(SV.DIALECT.spec(inner)))  # noqa: E731
assert text(call("f", first_match(b("+", y, b("+", 1, 2))), SV.concatenation(b("+", v("01"), v("01"))))) == (
    "f(first_match(y + (1 + 2)), 2'b10)")
assert text(call("f", first_match(SV.conditional(1, y, 2)))) == "f(first_match(1 ? y : 2))"
# A residue does not keep what does not depend on it from folding around it.
assert text(b("+", lambda u_: u_.incrementexpression(lambda r: r.operator("++").operand(x)), b("+", 1, 2))) == "++x + (1 + 2)"
assert text(SV.concatenation(lambda u_: u_.incrementexpression(lambda r: r.operator("++").operand(x)), b("+", v("01"), v("01")))) == "{++x, 2'b10}"

<a id="SVS-15"></a>
## SVS-15 · A value whose type no literal has (2-state, `shortreal`) folds into a cast of a literal to its type, where its states could matter

In [ ]:
# Where a 2-state value's states, or a `shortreal`'s format, could change what stays around it, it folds into a cast of
# a literal to its type (6.24.1), which keeps them: a closed constructor expression, as a literal is.
assert text(SV.concatenation(y, x), {"x": SVD.Logic(INT, 5)}) == "{y, int'(5)}"
assert text(SV.concatenation(y, SV.cast(atom("int"), -3))) == "{y, int'(-3)}"
assert text(SV.concatenation(y, SV.cast(atom("byte"), 300))) == "{y, byte'(8'sb00101100)}"
assert text(SV.concatenation(y, SV.cast(atom("shortint"), 1))) == "{y, shortint'(16'sb0000000000000001)}"
assert text(SV.concatenation(y, SV.cast(atom("longint"), 7))) == "{y, longint'(64'sb" + "0" * 61 + "111)}"
# A keyword's type of the other signedness is that cast's signing; a 2-state type as wide as no keyword's stays.
assert text(SV.concatenation(y, SV.cast(atom("int", "unsigned"), -1))) == "{y, unsigned'(int'(-1))}"
assert text(SV.concatenation(y, x), {"x": SVD.Logic(SVD.vector(1, True, 2), 1)}) == "{y, signed'(bit'(1'b1))}"
assert text(SV.concatenation(y, SV.cast(vector_type("bit", 3, 0), 5))) == "{y, bit [3:0]'(5)}"
assert text(SV.concatenation(y, b("+", x, x)), {"x": SVD.Logic(SVD.vector(4, False, 2), 5)}) == "{y, x + x}"
# A context-determined term folds into a cast to its context's type once that type is known; unknown, it stays, since
# a wider context could keep what its own type drops.
assert text(b("+", y, b("+", SV.cast(atom("int"), 1), SV.cast(atom("int"), 2))), SVV.Scope(types={"y": INT})) == "y + int'(3)"
assert text(b("+", y, b("+", SV.cast(atom("int"), 1), SV.cast(atom("int"), 2)))) == "y + (int'(1) + int'(2))"
assert text(SV.concatenation(y, b("+", SV.cast(atom("int"), 2147483647), SV.cast(atom("int"), 1)))) == "{y, int'(-2147483648)}"
# A `shortreal` is written by the fewest digits that a cast rounds to it, an infinity as a division.
shortreal_ = lambda operand: SV.cast(lambda t: t.nonintegertype(lambda n: n.keyword("shortreal")), operand)  # noqa: E731
assert text(b("+", y, call("$bitstoshortreal", v("00111111110000000000000000000000")))) == "y + shortreal'(1.5)"
assert text(b("+", y, shortreal_(b("/", 1.0, 3)))) == "y + shortreal'(0.33333334)"
assert text(b("+", y, call("$bitstoshortreal", v("01111111100000000000000000000000")))) == "y + shortreal'(1.0 / 0.0)"
# Within a sublanguage's term, which no rule reads, a term of such a type folds alone, into such a cast.
sequence = lambda inner: lambda u_: u_.firstmatchsequence(lambda r: r.sequence(SV.DIALECT.spec(inner)))  # noqa: E731
assert text(call("f", sequence(SV.cast(atom("int"), b("+", 1, 2))))) == "f(first_match(int'(3)))"
assert text(call("f", sequence(x)), {"x": True}) == "f(first_match(bit'(1'b1)))"
# Simplified again, a residual is itself; its root, whose context is no expression's, folds into a literal, as before.
residual = simplified(SV.concatenation(y, SV.cast(atom("int"), 5))).residual
assert text(residual) == "{y, int'(5)}"
# A known value that neither a literal nor a cast writes (an array) keeps its subterm, as in a simplifier that writes no
# closed constructor expression but a literal.
from mbse.Expressions.Framework import Simplifiers as FS
held = term(SV.concatenation(y, x))
for simplifier in (SVV.SIMPLIFIER, FS.Simplifier(SVV.RULES, literal=lambda _: None)):
    found = simplifier.known(held, [1, 2])
    assert found.expression is held and found.known and not found.constructed
assert simplified(SV.cast(atom("int"), 5)).literal and value(SV.cast(atom("int"), 5)) == "32'sb" + "0" * 29 + "101"

<a id="SVS-16"></a>
## SVS-16 · `$sformatf`'s `%u` and `%z` write a value's bits unformatted, and `%p` writes it as an assignment pattern (21.2.1)

In [ ]:
# `%u` writes a value's 2-state bits (x and z as 0) and `%z` its `aval` then its `bval` word, as VPI holds them: units of
# 32 bits, the least significant first, each little-endian; the string `$sformatf` gives holds no NUL byte (6.16).
assert written("%u", 0x41424344) == "DCBA" and written("%u", v("0100000101000010" * 3)) == "BABABA"
assert written("%u|%z", v("010000x1"), v("x1000001")) == "A|\xc1\x80" and written("%z", v("01000001xz")) == "\x06\x01\x03"
with raises(TypeError, match="%u expects an integral argument, got real"):
    written("%u", 1.5)
# `%p` writes an integral value as `%0d` does, a real as `%g` does, and an array as an assignment pattern of its items.
assert written("%p %p %p %p", 5, SV.cast(8, -1), v("10x1"), 1.5) == "5 -1 X 1.5" and written("%5p|%-5p|", 7, 7) == "    7|7    |"
assert written("%p", x, scope={"x": [1, [2.5, True], []]}) == "'{1, '{2.5, 1}, '{}}"
with raises(TypeError, match="$sformatf writes numbers and strings, not a list"):
    written("%d", x, scope={"x": [1]})
# A structure's members, and a class handle's, are their declaration's to write: `%p` of one stays (c).
for argument, scope in [(x, {"x": {"a": 1}}), (null, None)]:
    assert reasons(call("$sformatf", "%p", argument), scope) == [
        ("SystemVerilog systemcall '$sformatf' %p of an object is not simplified", "c")]